# 05. Análisis de componentes principales de la cinemática residual

Este cuaderno ajusta un PCA con las tres componentes locales de velocidad residual de 2019–2024. El PCA se construye sin usar el brillo; después se comparan las puntuaciones de los mismos ejes entre grupos de brillo y se guardan las tablas, figuras y parámetros necesarios para reproducir el análisis.


In [ ]:
# Bibliotecas utilizadas

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


## 1. Carga y contrato de entrada


In [ ]:
# Localizar la raíz del proyecto

ruta_actual = Path.cwd().resolve()

raiz_proyecto = next(
    (
        ruta
        for ruta in [ruta_actual, *ruta_actual.parents]
        if (ruta / "pyproject.toml").is_file()
    ),
    None,
)

if raiz_proyecto is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Abre VS Code o Jupyter desde el repositorio de la investigación."
    )

print("Raíz del proyecto:", raiz_proyecto)


In [ ]:
# Definir y comprobar la ruta de la base de entrenamiento

ruta_entrenamiento = (
    raiz_proyecto
    / "data"
    / "processed"
    / "geminidas_modelado_v2"
    / "geminidas_modelado_entrenamiento_2019_2024_v2.parquet"
)

if not ruta_entrenamiento.is_file():
    raise FileNotFoundError(
        "No se encontró la base de entrenamiento v2. "
        "Ejecuta primero el notebook 04."
    )

print("Archivo encontrado:", ruta_entrenamiento.name)


In [ ]:
# Cargar la base de entrenamiento y comprobar su partición

df_entrenamiento = pd.read_parquet(ruta_entrenamiento)

columnas_contrato = [
    "id_fila_origen",
    "source_year",
    "particion_modelado",
    "usar_pca",
    "grupo_brillo_ml",
    "delta_v_paralela_km_s",
    "delta_v_lambda_km_s",
    "delta_v_beta_km_s",
]

faltantes = [
    columna
    for columna in columnas_contrato
    if columna not in df_entrenamiento.columns
]

if faltantes:
    raise KeyError(
        f"Faltan columnas necesarias para el PCA: {faltantes}"
    )

if not df_entrenamiento["id_fila_origen"].is_unique:
    raise ValueError("La base de entrenamiento contiene IDs repetidos.")

if not df_entrenamiento["source_year"].between(2019, 2024).all():
    raise ValueError("La base del PCA contiene observaciones fuera de 2019–2024.")

if not (df_entrenamiento["particion_modelado"] == "entrenamiento").all():
    raise ValueError("La base del PCA contiene filas que no pertenecen a entrenamiento.")

print("Filas y columnas:", df_entrenamiento.shape)
print("Años presentes:", sorted(df_entrenamiento["source_year"].unique()))


## 2. Variables físicas del PCA


In [ ]:
# Seleccionar las tres componentes locales de velocidad residual

variables_pca = [
    "delta_v_paralela_km_s",
    "delta_v_lambda_km_s",
    "delta_v_beta_km_s",
]

datos_pca = df_entrenamiento.loc[
    df_entrenamiento["usar_pca"],
    variables_pca,
].copy()

if datos_pca.empty:
    raise ValueError("No hay trayectorias elegibles para el PCA.")

if not all(
    pd.api.types.is_numeric_dtype(datos_pca[columna])
    for columna in variables_pca
):
    raise TypeError("Las variables del PCA deben ser numéricas.")

if not np.isfinite(datos_pca.to_numpy(dtype=float)).all():
    raise ValueError("Las variables del PCA contienen valores no finitos.")

print("Trayectorias utilizables:", len(datos_pca))
print("Variables del PCA:", variables_pca)


## 3. Distribución de las componentes locales


In [ ]:
# Resumir las tres componentes antes de calcular el PCA

resumen_pca = datos_pca.describe().T

display(resumen_pca.round(4))

## 4. Centrado y matriz de covarianza


In [ ]:
# Construir la matriz de las tres componentes de velocidad

X = datos_pca.to_numpy()

n = X.shape[0]

# Calcular el promedio de cada columna
media = X.mean(axis=0)

# Centrar los datos restando sus promedios
X_c = X - media

print("Dimensiones de X:", X.shape)
print("Medias:", media)
print("Medias después de centrar:", X_c.mean(axis=0))

In [ ]:
# Calcular la matriz de covarianza

S = (X_c.T @ X_c) / (n - 1)

tabla_covarianza = pd.DataFrame(
    S,
    index=variables_pca,
    columns=variables_pca
)

display(tabla_covarianza.round(5))

# Verificar que la matriz sea simétrica
assert np.allclose(S, S.T)

## 5. Ajuste del PCA


In [ ]:
# Ajustar el PCA con las tres velocidades residuales de entrenamiento

pca = PCA(n_components=3)
pca.fit(X)

assert np.allclose(pca.mean_, media)
print("PCA ajustado con entrenamiento.")

In [ ]:
# Obtener la varianza explicada por cada componente

tabla_varianza = pd.DataFrame({
    "Componente": ["PC1", "PC2", "PC3"],
    "Varianza": pca.explained_variance_,
    "Porcentaje (%)": 100 * pca.explained_variance_ratio_,
    "Acumulado (%)": 100 * np.cumsum(pca.explained_variance_ratio_)
})

display(tabla_varianza.round(4))

In [ ]:
# Comprobar que el PCA conserva la varianza total

varianza_original = np.trace(S)
varianza_pca = pca.explained_variance_.sum()

assert np.isclose(varianza_original, varianza_pca)
assert np.isclose(pca.explained_variance_ratio_.sum(), 1)

print("Varianza total original:", round(varianza_original, 6))
print("Varianza total en PCA:", round(varianza_pca, 6))
print("Verificación correcta.")

## 6. Pesos y correlaciones de las componentes


In [ ]:
# Obtener los pesos de cada componente principal

pesos_pca = pd.DataFrame(
    pca.components_.T,
    index=variables_pca,
    columns=["PC1", "PC2", "PC3"]
)

display(pesos_pca.round(4))

# Comprobar que los ejes principales sean ortonormales
assert np.allclose(
    pca.components_ @ pca.components_.T,
    np.eye(3)
)

print("Los tres ejes son ortonormales.")

## 7. Puntuaciones de cada meteoro


In [ ]:
# Pasar las velocidades a coordenadas PC1, PC2 y PC3

coordenadas_pca = pca.transform(X)

df_pca = pd.DataFrame(
    coordenadas_pca,
    columns=["PC1", "PC2", "PC3"],
    index=datos_pca.index
)

# Conservar el identificador para integrar las puntuaciones con otros notebooks
df_pca_por_id = df_pca.copy()
df_pca_por_id.insert(
    0, "id_fila_origen",
    df_entrenamiento.loc[datos_pca.index, "id_fila_origen"].to_numpy()
)
assert df_pca_por_id["id_fila_origen"].is_unique
assert np.allclose(coordenadas_pca, X_c @ pca.components_.T)

display(df_pca_por_id.head())
print("Dimensiones:", df_pca_por_id.shape)

In [ ]:
# Comprobar las coordenadas y su matriz de covarianza

S_pca = np.cov(coordenadas_pca, rowvar=False)

display(pd.DataFrame(
    S_pca,
    index=["PC1", "PC2", "PC3"],
    columns=["PC1", "PC2", "PC3"]
).round(6))

# Verificar que las componentes no estén correlacionadas linealmente
assert np.allclose(
    S_pca,
    np.diag(pca.explained_variance_)
)

print("Covarianza verificada.")

## 8. Mapa de correlaciones


In [ ]:
# Calcular las correlaciones entre variables físicas y componentes principales

desviaciones = datos_pca.std(ddof=1).to_numpy()

cargas = (
    pca.components_.T
    * np.sqrt(pca.explained_variance_)
    / desviaciones[:, None]
)

tabla_cargas = pd.DataFrame(
    cargas,
    index=variables_pca,
    columns=["PC1", "PC2", "PC3"]
)

display(tabla_cargas.round(4))

# Comprobar que las correlaciones estén entre -1 y 1
assert np.abs(cargas).max() <= 1 + 1e-10

# Preparar las carpetas donde se guardarán resultados reproducibles
ruta_tablas = raiz_proyecto / "reports" / "tables"
ruta_figuras = raiz_proyecto / "reports" / "figures"

ruta_tablas.mkdir(parents=True, exist_ok=True)
ruta_figuras.mkdir(parents=True, exist_ok=True)


In [ ]:
# Visualizar las correlaciones entre variables y componentes

fig, ax = plt.subplots(figsize=(7, 4))

mapa = ax.imshow(
    cargas,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

ax.set_xticks(range(3), ["PC1", "PC2", "PC3"])

ax.set_yticks(
    range(3),
    ["Paralela", "Longitudinal", "Latitudinal"]
)

# Mostrar el valor numérico dentro de cada celda
for i in range(3):
    for j in range(3):
        ax.text(
            j, i, f"{cargas[i, j]:.3f}",
            ha="center",
            va="center",
            color="white" if abs(cargas[i, j]) > 0.5 else "black"
        )

barra = fig.colorbar(mapa, ax=ax)
barra.set_label("Correlación")

ax.set_xlabel("Componente principal")
ax.set_ylabel("Componente física de velocidad")
ax.set_title("Correlaciones entre variables físicas y PCA")

plt.tight_layout()

fig.savefig(
    ruta_figuras / "pca_mapa_correlaciones_2019_2024.pdf",
    bbox_inches="tight",
)
fig.savefig(
    ruta_figuras / "pca_mapa_correlaciones_2019_2024.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


## 9. Varianza explicada


In [ ]:
# Mostrar la varianza explicada sin repetir las tablas anteriores

fig, ax = plt.subplots(figsize=(6, 3.8))
porcentajes = 100 * pca.explained_variance_ratio_
ax.plot([1, 2, 3], porcentajes, marker="o", linewidth=1.5)
ax.set_xticks([1, 2, 3], ["PC1", "PC2", "PC3"])
ax.set_xlabel("Componente principal")
ax.set_ylabel("Varianza explicada (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.2)
ax.set_title("Scree plot — entrenamiento 2019–2024")
plt.tight_layout()

fig.savefig(
    ruta_figuras / "pca_scree_2019_2024.pdf",
    bbox_inches="tight",
)
fig.savefig(
    ruta_figuras / "pca_scree_2019_2024.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


## 10. Proyección PC1–PC2


In [ ]:
# Representar la concentración de meteoros en PC1 y PC2

fig, ax = plt.subplots(figsize=(8, 6))

mapa = ax.hexbin(
    df_pca["PC1"],
    df_pca["PC2"],
    gridsize=55,
    mincnt=1,
    bins="log",
    cmap="viridis"
)

barra = fig.colorbar(mapa, ax=ax)
barra.set_label("Número de meteoros (escala logarítmica)")

ax.set_xlabel("PC1 (km/s)")
ax.set_ylabel("PC2 (km/s)")
ax.set_title("Distribución cinemática de las Gemínidas (2019–2024)")

ax.set_aspect("equal", adjustable="box")

plt.tight_layout()

fig.savefig(
    ruta_figuras / "pca_proyeccion_pc1_pc2_2019_2024.pdf",
    bbox_inches="tight",
)
fig.savefig(
    ruta_figuras / "pca_proyeccion_pc1_pc2_2019_2024.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


## 11. Biplot


In [ ]:
# Biplot: nube PC1-PC2 con las flechas de las variables originales

fig, ax = plt.subplots(figsize=(9, 5.3))
mapa = ax.hexbin(
    df_pca["PC1"], df_pca["PC2"],
    gridsize=55, mincnt=1, bins="log", cmap="viridis"
)

# Las correlaciones se reescalan solo para el dibujo
nombres = ["Paralela", "Longitudinal", "Latitudinal"]
colores = ["firebrick", "darkorange", "purple"]
escala_flechas = 0.65

for i, (nombre, color) in enumerate(zip(nombres, colores)):
    x = cargas[i, 0] * escala_flechas
    y = cargas[i, 1] * escala_flechas
    ax.annotate("", xy=(x, y), xytext=(0, 0),
                arrowprops={"arrowstyle": "->", "color": color, "lw": 2})
    ax.plot([], [], color=color, linewidth=2, label=nombre)

ax.set_xlabel("PC1 (km/s)")
ax.set_ylabel("PC2 (km/s)")
ax.set_aspect("equal", adjustable="box")
ax.set_title("Velocidades residuales y variables físicas")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.32),
          ncol=3, frameon=False)
fig.colorbar(mapa, ax=ax, label="Número de meteoros (escala logarítmica)")
fig.text(0.5, 0.015, "Flechas de correlación reescaladas por un factor visual de 0.65.",
         ha="center", fontsize=9)
fig.tight_layout(rect=[0, 0.14, 1, 1])
fig.savefig(
    ruta_figuras / "pca_biplot_2019_2024.pdf",
    bbox_inches="tight",
)
fig.savefig(
    ruta_figuras / "pca_biplot_2019_2024.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


## 12. Comparación según brillo


In [ ]:
# Incorporar el grupo de brillo después de ajustar el PCA

df_analisis = df_pca.copy()

df_analisis["grupo_brillo"] = df_entrenamiento.loc[
    datos_pca.index,
    "grupo_brillo_ml",
].astype("string")

orden_grupos = ["brillante", "intermedio", "tenue"]

conteos_grupos = (
    df_analisis["grupo_brillo"]
    .value_counts(dropna=False)
)

print(conteos_grupos)

resumen_grupos = (
    df_analisis.loc[
        df_analisis["grupo_brillo"].isin(orden_grupos)
    ]
    .groupby("grupo_brillo")[["PC1", "PC2", "PC3"]]
    .agg(["mean", "std"])
    .reindex(orden_grupos)
)

display(resumen_grupos.round(4))


In [ ]:
# Comparar las distribuciones acumuladas de PC1 según brillo

fig, ax = plt.subplots(figsize=(8, 5))

grupos = orden_grupos

for grupo in grupos:

    valores = df_analisis.loc[
        df_analisis["grupo_brillo"] == grupo, "PC1"
    ].sort_values().to_numpy()

    acumulada = np.arange(1, len(valores) + 1) / len(valores)

    ax.plot(valores, acumulada, label=grupo.capitalize())

ax.set_xlabel("PC1 (km/s)")
ax.set_ylabel("Proporción acumulada")
ax.set_title("Distribución de PC1 según el brillo")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

fig.savefig(
    ruta_figuras / "pca_ecdf_pc1_por_brillo_2019_2024.pdf",
    bbox_inches="tight",
)
fig.savefig(
    ruta_figuras / "pca_ecdf_pc1_por_brillo_2019_2024.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


## 13. Centros y dispersión por grupo


In [ ]:
# Comparar centros y dispersiones de los grupos en PC1 y PC2

fig, ax = plt.subplots(figsize=(8, 5))

grupos = orden_grupos

angulos = np.linspace(0, 2 * np.pi, 200)
circulo = np.array([np.cos(angulos), np.sin(angulos)])

for grupo in grupos:

    datos = df_analisis.loc[
        df_analisis["grupo_brillo"] == grupo,
        ["PC1", "PC2"]
    ].to_numpy()

    # Centro y matriz de covarianza del grupo
    centro = datos.mean(axis=0)
    covarianza = np.cov(datos, rowvar=False)

    # Direcciones y varianzas principales del grupo
    valores, vectores = np.linalg.eigh(covarianza)

    # Construir la elipse de una desviación estándar
    elipse = centro[:, None] + vectores @ (
        np.sqrt(valores)[:, None] * circulo
    )

    linea, = ax.plot(
        elipse[0], elipse[1],
        label=grupo.capitalize()
    )

    ax.scatter(
        centro[0], centro[1],
        color=linea.get_color(),
        marker="x",
        s=60
    )

ax.set_xlabel("PC1 (km/s)")
ax.set_ylabel("PC2 (km/s)")
ax.set_title("Elipses de covarianza por grupo de brillo")
ax.set_aspect("equal", adjustable="box")
# Colocar la leyenda fuera del área de datos
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
    frameon=False
)
ax.grid(alpha=0.2)
plt.tight_layout()

fig.savefig(
    ruta_figuras / "pca_elipses_covarianza_por_brillo_2019_2024.pdf",
    bbox_inches="tight",
)
fig.savefig(
    ruta_figuras / "pca_elipses_covarianza_por_brillo_2019_2024.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


## 14. Dispersión en las tres componentes


In [ ]:
# Comparar la dispersión completa en los tres ejes del PCA global

componentes = ["PC1", "PC2", "PC3"]
grupos = orden_grupos

resultados = []

for grupo in grupos:

    datos_grupo = df_analisis.loc[
        df_analisis["grupo_brillo"] == grupo,
        componentes
    ]

    # Matriz de covarianza del grupo
    S_grupo = datos_grupo.cov()

    # Suma de las tres varianzas
    dispersion_total = np.trace(S_grupo.to_numpy())

    resultados.append({
        "Grupo": grupo,
        "Meteoros": len(datos_grupo),
        "Varianza PC1": S_grupo.loc["PC1", "PC1"],
        "Varianza PC2": S_grupo.loc["PC2", "PC2"],
        "Varianza PC3": S_grupo.loc["PC3", "PC3"],
        "Dispersión total": dispersion_total
    })

tabla_dispersion = pd.DataFrame(resultados)

display(tabla_dispersion.round(4))


## 15. Covarianzas y archivos de salida


In [ ]:
# Calcular las matrices de covarianza por grupo sin rehacer el PCA

componentes = ["PC1", "PC2", "PC3"]
covarianzas_grupos = []

for grupo in orden_grupos:
    datos_grupo = df_analisis.loc[
        df_analisis["grupo_brillo"] == grupo,
        componentes,
    ]

    cov_grupo = datos_grupo.cov()

    if len(datos_grupo) <= 1:
        raise ValueError(
            f"No hay suficientes observaciones en el grupo {grupo}."
        )

    if not np.allclose(cov_grupo, cov_grupo.T):
        raise ValueError(
            f"La covarianza del grupo {grupo} no es simétrica."
        )

    if np.linalg.eigvalsh(cov_grupo).min() < -1e-10:
        raise ValueError(
            f"La covarianza del grupo {grupo} no es semidefinida positiva."
        )

    print(f"\n{grupo.capitalize()} — {len(datos_grupo)} meteoros")
    display(cov_grupo.round(5))

    for fila in componentes:
        for columna in componentes:
            covarianzas_grupos.append(
                {
                    "grupo": grupo,
                    "fila": fila,
                    "columna": columna,
                    "covarianza": cov_grupo.loc[fila, columna],
                }
            )

tabla_covarianzas_grupos = pd.DataFrame(covarianzas_grupos)


In [ ]:
# Guardar puntuaciones, parámetros, tablas y resúmenes del PCA

ruta_puntuaciones = (
    ruta_tablas
    / "pca_entrenamiento_2019_2024.parquet"
)

ruta_parametros = (
    ruta_tablas
    / "pca_parametros_2019_2024.npz"
)

ruta_varianza = (
    ruta_tablas
    / "pca_varianza_explicada_2019_2024.csv"
)

ruta_pesos = (
    ruta_tablas
    / "pca_pesos_2019_2024.csv"
)

ruta_correlaciones = (
    ruta_tablas
    / "pca_correlaciones_2019_2024.csv"
)

ruta_dispersion = (
    ruta_tablas
    / "pca_dispersion_por_brillo_2019_2024.csv"
)

ruta_covarianzas = (
    ruta_tablas
    / "pca_covarianzas_por_brillo_2019_2024.csv"
)

# Puntuaciones individuales
if not df_pca_por_id["id_fila_origen"].is_unique:
    raise ValueError("Las puntuaciones PCA contienen IDs repetidos.")

df_pca_por_id.to_parquet(
    ruta_puntuaciones,
    index=False,
)

# Parámetros necesarios para reproducir la transformación
np.savez_compressed(
    ruta_parametros,
    variables=np.array(variables_pca),
    media=pca.mean_,
    componentes=pca.components_,
    varianza=pca.explained_variance_,
    proporcion_varianza=pca.explained_variance_ratio_,
    anios_entrenamiento=np.arange(2019, 2025),
    unidades=np.array(["km/s", "km/s", "km/s"]),
)

# Tablas compactas
tabla_varianza.to_csv(ruta_varianza, index=False)

(
    pesos_pca
    .rename_axis("variable")
    .reset_index()
    .to_csv(ruta_pesos, index=False)
)

(
    tabla_cargas
    .rename_axis("variable")
    .reset_index()
    .to_csv(ruta_correlaciones, index=False)
)

tabla_dispersion.to_csv(
    ruta_dispersion,
    index=False,
)

tabla_covarianzas_grupos.to_csv(
    ruta_covarianzas,
    index=False,
)

# Comprobar que las puntuaciones y parámetros guardados reproducen el PCA
puntuaciones_guardadas = pd.read_parquet(
    ruta_puntuaciones
)

if not puntuaciones_guardadas[
    "id_fila_origen"
].equals(
    df_pca_por_id[
        "id_fila_origen"
    ].reset_index(drop=True)
):
    raise ValueError(
        "Los identificadores guardados del PCA no coinciden."
    )

if not np.allclose(
    puntuaciones_guardadas[["PC1", "PC2", "PC3"]],
    coordenadas_pca,
):
    raise ValueError(
        "Las puntuaciones PCA guardadas no coinciden con las originales."
    )

with np.load(
    ruta_parametros,
    allow_pickle=False,
) as parametros:

    puntuaciones_reconstruidas = (
        X - parametros["media"]
    ) @ parametros["componentes"].T

    if list(parametros["variables"]) != variables_pca:
        raise ValueError(
            "El orden de variables guardado no coincide con el PCA."
        )

    if not np.allclose(
        puntuaciones_reconstruidas,
        coordenadas_pca,
    ):
        raise ValueError(
            "Los parámetros guardados no reproducen las puntuaciones PCA."
        )

print("Puntuaciones:", ruta_puntuaciones.relative_to(raiz_proyecto))
print("Parámetros:", ruta_parametros.relative_to(raiz_proyecto))
print("Tablas y figuras guardadas en reports/.")
print("Comprobado: el PCA puede reproducirse sin reajustarlo.")
